In [15]:
import pandas as pd
import numpy as np
import statsmodels.formula.api as smf
from scipy import stats

In [16]:
df = pd.read_csv("/Users/hugocather/MD4002 Dissertation/2024/2024_Dataset_Cleaned(List_size).csv")

In [20]:
# Renaming the columns for easier use in formulas
df = df.rename(columns={
    'Elderly (65+) - % of Total': 'Elderly_Pct',
    'Young (0-14) - % of Total': 'Young_Pct',
    'Working Age (15-64) - % of Total': 'WorkingAge_Pct',
    'Total_DDDs_Per_1000_Registrants': 'Total_DDDs',
    'Total_Strong_Opioid_DDDs_Per_1000_Registrants': 'Strong_DDDs',
    'Total_Weak_Opioid_DDDs_Per_1000_Registrants': 'Weak_DDDs'
})


In [21]:
# Drop any rows with missing values in key variables
model_vars = ['Log_Total_DDDs', 'Weighted_SIMD', 'Elderly_Pct',
              'Overall_Pct_Female', 'Clinical_Need_Composite',
              'PracticeListSize', 'HBT']
df_model = df.dropna(subset=model_vars).copy()

In [22]:
print(f"Rows before: {len(df)}")
print(f"Rows after:  {len(df_model)}")
print(f"Rows dropped: {len(df) - len(df_model)}")

# See which variables had missing values
for col in model_vars:
    missing = df[col].isna().sum()
    if missing > 0:
        print(f"  {col}: {missing} missing")

Rows before: 807
Rows after:  807
Rows dropped: 0


In [23]:
# Standardise continuous predictors to enable comparison of coefficients (as the predictors are on very different scales)
# This means a one-unit change in the model = one SD change in the predictor, making it easier to compare the relative importance of different variables
for col in ['Weighted_SIMD', 'Elderly_Pct', 'Overall_Pct_Female',
            'Clinical_Need_Composite', 'PracticeListSize']:
    df_model[f'{col}_z'] = (df_model[col] - df_model[col].mean()) / df_model[col].std()

print(f"Practices in model: {len(df_model)}")
print(f"NHS Boards: {df_model['HBT'].nunique()}")
print(f"Practices per Board: {df_model.groupby('HBT').size().min()} - {df_model.groupby('HBT').size().max()}")

Practices in model: 807
NHS Boards: 14
Practices per Board: 6 - 207


In [24]:
print("\n" + "=" * 80)
print("MODEL 0: NULL MODEL (Random Intercept Only)")
print("=" * 80)

model0 = smf.mixedlm(
    "Log_Total_DDDs ~ 1",
    data=df_model,
    groups="HBT"
)
result0 = model0.fit(reml=True)
print(result0.summary())

# Extract variance components
var_board_0 = result0.cov_re.iloc[0, 0]  # Between-Board variance
var_resid_0 = result0.scale              # Within-Board (residual) variance
icc_0 = var_board_0 / (var_board_0 + var_resid_0)

print(f"\n--- Variance Decomposition ---")
print(f"Between-Board variance (σ²_u): {var_board_0:.4f}")
print(f"Within-Board variance (σ²_e):  {var_resid_0:.4f}")
print(f"Total variance:                {var_board_0 + var_resid_0:.4f}")
print(f"ICC: {icc_0:.4f} ({icc_0*100:.1f}%)")
print(f"\nInterpretation: {icc_0*100:.1f}% of the variance in log opioid prescribing")
print(f"rates is attributable to differences between NHS Boards.")


# 


MODEL 0: NULL MODEL (Random Intercept Only)
           Mixed Linear Model Regression Results
Model:            MixedLM Dependent Variable: Log_Total_DDDs
No. Observations: 807     Method:             REML          
No. Groups:       14      Scale:              0.2024        
Min. group size:  6       Log-Likelihood:     -525.4248     
Max. group size:  207     Converged:          Yes           
Mean group size:  57.6                                      
-------------------------------------------------------------
               Coef.  Std.Err.    z     P>|z|  [0.025  0.975]
-------------------------------------------------------------
Intercept      7.098     0.111  64.057  0.000   6.881   7.316
HBT Var        0.163     0.153                               


--- Variance Decomposition ---
Between-Board variance (σ²_u): 0.1628
Within-Board variance (σ²_e):  0.2024
Total variance:                0.3652
ICC: 0.4458 (44.6%)

Interpretation: 44.6% of the variance in log opioid prescribin

In [25]:
print("\n" + "=" * 80)
print("MODEL 1: DEPRIVATION (Weighted SIMD)")
print("=" * 80)

model1 = smf.mixedlm(
    "Log_Total_DDDs ~ Weighted_SIMD_z",
    data=df_model,
    groups="HBT"
)
result1 = model1.fit(reml=True)
print(result1.summary())

var_board_1 = result1.cov_re.iloc[0, 0]
var_resid_1 = result1.scale

# Proportional Change in Variance compared to null model
pcv_board_1 = (var_board_0 - var_board_1) / var_board_0
pcv_resid_1 = (var_resid_0 - var_resid_1) / var_resid_0
icc_1 = var_board_1 / (var_board_1 + var_resid_1)

print(f"\n--- Variance Decomposition ---")
print(f"Between-Board variance (σ²_u): {var_board_1:.4f}")
print(f"Within-Board variance (σ²_e):  {var_resid_1:.4f}")
print(f"ICC: {icc_1:.4f} ({icc_1*100:.1f}%)")
print(f"\n--- Proportional Change in Variance (vs Null) ---")
print(f"Board-level PCV:    {pcv_board_1:.4f} ({pcv_board_1*100:.1f}% of Board variance explained)")
print(f"Practice-level PCV: {pcv_resid_1:.4f} ({pcv_resid_1*100:.1f}% of practice variance explained)")



MODEL 1: DEPRIVATION (Weighted SIMD)
           Mixed Linear Model Regression Results
Model:            MixedLM Dependent Variable: Log_Total_DDDs
No. Observations: 807     Method:             REML          
No. Groups:       14      Scale:              0.1286        
Min. group size:  6       Log-Likelihood:     -344.8594     
Max. group size:  207     Converged:          Yes           
Mean group size:  57.6                                      
------------------------------------------------------------
                 Coef.  Std.Err.    z    P>|z| [0.025 0.975]
------------------------------------------------------------
Intercept         7.148    0.085  84.346 0.000  6.982  7.314
Weighted_SIMD_z  -0.313    0.015 -21.533 0.000 -0.341 -0.284
HBT Var           0.095    0.114                            


--- Variance Decomposition ---
Between-Board variance (σ²_u): 0.0946
Within-Board variance (σ²_e):  0.1286
ICC: 0.4239 (42.4%)

--- Proportional Change in Variance (vs Null) ---
B

In [26]:
print("\n" + "=" * 80)
print("MODEL 2: DEPRIVATION + DEMOGRAPHICS (Elderly %, Female %)")
print("=" * 80)

model2 = smf.mixedlm(
    "Log_Total_DDDs ~ Weighted_SIMD_z + Elderly_Pct_z + Overall_Pct_Female_z",
    data=df_model,
    groups="HBT"
)
result2 = model2.fit(reml=True)
print(result2.summary())

var_board_2 = result2.cov_re.iloc[0, 0]
var_resid_2 = result2.scale

pcv_board_2 = (var_board_0 - var_board_2) / var_board_0
pcv_resid_2 = (var_resid_0 - var_resid_2) / var_resid_0
icc_2 = var_board_2 / (var_board_2 + var_resid_2)

print(f"\n--- Variance Decomposition ---")
print(f"Between-Board variance (σ²_u): {var_board_2:.4f}")
print(f"Within-Board variance (σ²_e):  {var_resid_2:.4f}")
print(f"ICC: {icc_2:.4f} ({icc_2*100:.1f}%)")
print(f"\n--- Proportional Change in Variance (vs Null) ---")
print(f"Board-level PCV:    {pcv_board_2:.4f} ({pcv_board_2*100:.1f}% of Board variance explained)")
print(f"Practice-level PCV: {pcv_resid_2:.4f} ({pcv_resid_2*100:.1f}% of practice variance explained)")


MODEL 2: DEPRIVATION + DEMOGRAPHICS (Elderly %, Female %)
             Mixed Linear Model Regression Results
Model:              MixedLM  Dependent Variable:  Log_Total_DDDs
No. Observations:   807      Method:              REML          
No. Groups:         14       Scale:               0.1126        
Min. group size:    6        Log-Likelihood:      -299.5157     
Max. group size:    207      Converged:           Yes           
Mean group size:    57.6                                        
----------------------------------------------------------------
                     Coef.  Std.Err.    z    P>|z| [0.025 0.975]
----------------------------------------------------------------
Intercept             7.088    0.097  73.207 0.000  6.898  7.277
Weighted_SIMD_z      -0.374    0.015 -25.225 0.000 -0.403 -0.345
Elderly_Pct_z         0.151    0.017   8.958 0.000  0.118  0.184
Overall_Pct_Female_z  0.051    0.013   3.895 0.000  0.025  0.077
HBT Var               0.125    0.159         

In [27]:
print("\n" + "=" * 80)
print("MODEL 3: FULL MODEL (SIMD + Elderly + Female + Clinical Need + List Size)")
print("=" * 80)

model3 = smf.mixedlm(
    "Log_Total_DDDs ~ Weighted_SIMD_z + Elderly_Pct_z + Overall_Pct_Female_z + Clinical_Need_Composite_z + PracticeListSize_z",
    data=df_model,
    groups="HBT"
)
result3 = model3.fit(reml=True)
print(result3.summary())

var_board_3 = result3.cov_re.iloc[0, 0]
var_resid_3 = result3.scale

pcv_board_3 = (var_board_0 - var_board_3) / var_board_0
pcv_resid_3 = (var_resid_0 - var_resid_3) / var_resid_0
icc_3 = var_board_3 / (var_board_3 + var_resid_3)

print(f"\n--- Variance Decomposition ---")
print(f"Between-Board variance (σ²_u): {var_board_3:.4f}")
print(f"Within-Board variance (σ²_e):  {var_resid_3:.4f}")
print(f"ICC: {icc_3:.4f} ({icc_3*100:.1f}%)")
print(f"\n--- Proportional Change in Variance (vs Null) ---")
print(f"Board-level PCV:    {pcv_board_3:.4f} ({pcv_board_3*100:.1f}% of Board variance explained)")
print(f"Practice-level PCV: {pcv_resid_3:.4f} ({pcv_resid_3*100:.1f}% of practice variance explained)")




MODEL 3: FULL MODEL (SIMD + Elderly + Female + Clinical Need + List Size)
                Mixed Linear Model Regression Results
Model:               MixedLM    Dependent Variable:    Log_Total_DDDs
No. Observations:    807        Method:                REML          
No. Groups:          14         Scale:                 0.1111        
Min. group size:     6          Log-Likelihood:        -299.4450     
Max. group size:     207        Converged:             Yes           
Mean group size:     57.6                                            
---------------------------------------------------------------------
                          Coef.  Std.Err.    z    P>|z| [0.025 0.975]
---------------------------------------------------------------------
Intercept                  7.092    0.093  76.004 0.000  6.909  7.275
Weighted_SIMD_z           -0.382    0.016 -23.494 0.000 -0.414 -0.350
Elderly_Pct_z              0.167    0.022   7.597 0.000  0.124  0.210
Overall_Pct_Female_z       0.04

In [28]:
print("\n" + "=" * 80)
print("MODEL COMPARISON SUMMARY")
print("=" * 80)

summary_data = {
    'Model': ['M0: Null', 'M1: SIMD', 'M2: + Demographics', 'M3: Full'],
    'Board Var (σ²_u)': [var_board_0, var_board_1, var_board_2, var_board_3],
    'Resid Var (σ²_e)': [var_resid_0, var_resid_1, var_resid_2, var_resid_3],
    'ICC': [icc_0, icc_1, icc_2, icc_3],
    'Board PCV (%)': [0, pcv_board_1 * 100, pcv_board_2 * 100, pcv_board_3 * 100],
    'Resid PCV (%)': [0, pcv_resid_1 * 100, pcv_resid_2 * 100, pcv_resid_3 * 100],
    'AIC': [result0.aic, result1.aic, result2.aic, result3.aic],
    'BIC': [result0.bic, result1.bic, result2.bic, result3.bic]
}

summary_df = pd.DataFrame(summary_data)
print(summary_df.round(4).to_string(index=False))


MODEL COMPARISON SUMMARY
             Model  Board Var (σ²_u)  Resid Var (σ²_e)    ICC  Board PCV (%)  Resid PCV (%)  AIC  BIC
          M0: Null            0.1628            0.2024 0.4458         0.0000         0.0000  NaN  NaN
          M1: SIMD            0.0946            0.1286 0.4239        41.8875        36.4713  NaN  NaN
M2: + Demographics            0.1253            0.1126 0.5268        23.0239        44.3795  NaN  NaN
          M3: Full            0.1161            0.1111 0.5109        28.7138        45.0966  NaN  NaN


In [29]:
for outcome, label in [('Log_Strong_DDDs', 'STRONG OPIOIDS'),
                       ('Log_Weak_DDDs', 'WEAK OPIOIDS')]:

    print("\n" + "=" * 80)
    print(f"SENSITIVITY ANALYSIS: FULL MODEL — {label}")
    print("=" * 80)

    model_sens = smf.mixedlm(
        f"{outcome} ~ Weighted_SIMD_z + Elderly_Pct_z + Overall_Pct_Female_z + Clinical_Need_Composite_z + PracticeListSize_z",
        data=df_model,
        groups="HBT"
    )
    result_sens = model_sens.fit(reml=True)
    print(result_sens.summary())

    var_board_s = result_sens.cov_re.iloc[0, 0]
    var_resid_s = result_sens.scale
    icc_s = var_board_s / (var_board_s + var_resid_s)

    print(f"\n--- Variance Decomposition ---")
    print(f"Between-Board variance: {var_board_s:.4f}")
    print(f"Within-Board variance:  {var_resid_s:.4f}")
    print(f"ICC: {icc_s:.4f} ({icc_s*100:.1f}%)")

    # Coefficient table with exponentiated values
    coef_sens = pd.DataFrame({
        'Coefficient': result_sens.fe_params,
        'Std Error': result_sens.bse_fe,
        'p-value': result_sens.pvalues,
        'Exp(Coef)': np.exp(result_sens.fe_params),
        '% Change per 1 SD': (np.exp(result_sens.fe_params) - 1) * 100
    })
    print(f"\nCoefficient Table ({label}):")
    print(coef_sens.round(4).to_string())


SENSITIVITY ANALYSIS: FULL MODEL — STRONG OPIOIDS
                Mixed Linear Model Regression Results
Model:                MixedLM   Dependent Variable:   Log_Strong_DDDs
No. Observations:     807       Method:               REML           
No. Groups:           14        Scale:                0.1705         
Min. group size:      6         Log-Likelihood:       -463.1830      
Max. group size:      207       Converged:            Yes            
Mean group size:      57.6                                           
---------------------------------------------------------------------
                          Coef.  Std.Err.    z    P>|z| [0.025 0.975]
---------------------------------------------------------------------
Intercept                  5.332    0.063  84.330 0.000  5.209  5.456
Weighted_SIMD_z           -0.392    0.020 -19.589 0.000 -0.431 -0.353
Elderly_Pct_z              0.227    0.027   8.463 0.000  0.174  0.280
Overall_Pct_Female_z       0.018    0.017   1.099 0.272